<a href="https://colab.research.google.com/github/zeyneptass/vector-databases-and-rag-fundamentals/blob/main/SemanticSearchDemo2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pinecone sentence-transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 15.9 MB/s eta 0:00:00


In [ ]:
import warnings
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

In [ ]:
from google.colab import userdata
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')

pc = Pinecone(api_key=PINECONE_API_KEY)

In [ ]:
from datasets import load_dataset

# Hatalı kullanım:
# dataset = load_dataset("ag_news", split="train[:2000]")

# Doğru kullanım (namespace'li):
dataset = load_dataset("fancyzhx/ag_news", split="train[:2000]")

questions = [item['text'] for item in dataset]
print(f"Toplam {len(questions)} haber yüklendi.")
print("Örnek:", questions[0])


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Toplam 2000 haber yüklendi.
Örnek: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model boyutu: {model.get_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model boyutu: 384


In [ ]:
from pinecone import Pinecone, ServerlessSpec
from google.colab import userdata

# API anahtarını al
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "semantic-search-demo2"

# Varsa eski dizini sil
if index_name in pc.list_indexes().names():
    print(f"'{index_name}' zaten var, siliniyor...")
    pc.delete_index(index_name)

# Yeni dizin oluştur
pc.create_index(
    name=index_name,
    dimension=model.get_embedding_dimension(),
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)
print(f"'{index_name}' dizini oluşturuldu.")

'semantic-search-demo2' dizini oluşturuldu.


In [ ]:
import time

while not pc.describe_index(index_name).status.ready:
    print("Dizin hazırlanıyor...")
    time.sleep(2)

print("Dizin kullanıma hazır!")

Dizin kullanıma hazır!


In [ ]:
index = pc.Index(index_name)
print(index)

Index(host='https://semantic-search-demo2-h0jgoc3.svc.aped-4627-b74a.pinecone.io')


In [ ]:
from tqdm.auto import tqdm

batch_size = 100

for i in tqdm(range(0, len(questions), batch_size)):
    # Bu partideki haberleri al
    batch = questions[i:i + batch_size]

    # Haberleri vektörlere çevir
    embeddings = model.encode(batch).tolist()

    # Pinecone formatına hazırla: (id, vektör, metadata)
    vectors_to_upsert = []
    for j, (text, embedding) in enumerate(zip(batch, embeddings)):
        vector_id = f"news_{i + j}"
        metadata = {"text": text}
        vectors_to_upsert.append((vector_id, embedding, metadata))

    # Pinecone'a yükle
    index.upsert(vectors=vectors_to_upsert)

print("Tüm haberler Pinecone'a yüklendi!")

  0%|          | 0/20 [00:00<?, ?it/s]

Tüm haberler Pinecone'a yüklendi!


In [ ]:
stats = index.describe_index_stats()
print(f"Toplam vektör sayısı: {stats['total_vector_count']}")

Toplam vektör sayısı: 2000


In [ ]:
def run_query(query_text, top_k=5):
    """
    Bir sorgu metnini alır, vektöre çevirir ve Pinecone'da arar.
    """
    # Sorguyu vektöre çevir
    query_embedding = model.encode(query_text).tolist()

    # Pinecone'da ara
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    # Sonuçları yazdır
    print(f"\nSorgu: '{query_text}'")
    print("-" * 70)
    for match in results['matches']:
        score = match['score']
        text = match['metadata']['text'][:120]  # İlk 120 karakter
        print(f"Benzerlik: {score:.4f} | {text}...")
    print("-" * 70)

    return results

In [ ]:
# Test 1: Teknoloji haberleri
run_query("What is the latest technology news?")

# Test 2: Spor haberleri
run_query("Who won the football match?")

# Test 3: İş dünyası haberleri
run_query("Stock market prices today")

# Test 4: Bilim haberleri
run_query("New scientific discovery")


Sorgu: 'What is the latest technology news?'
----------------------------------------------------------------------
Benzerlik: 0.4871 | Group to Propose New High-Speed Wireless Format (Reuters) Reuters - A group of technology companies\including Texas Inst...
Benzerlik: 0.4853 | The  #39;i #39; in  #39;Internet #39; It #39;s not exactly an earth-shaking development but I expect that Wired News #39...
Benzerlik: 0.4559 | News: Technology Already Exists to Stabilize Global Warming Existing technologies could stop the escalation of global wa...
Benzerlik: 0.4337 | Tech market indeed soft, but HP woes one of a kind HP and Cisco spooked investors with recent comments about sales, but ...
Benzerlik: 0.4180 | The Internet Is Calling Web phone service is exploding. And maverick Jeffrey Citron lit the industry's fuse...
----------------------------------------------------------------------

Sorgu: 'Who won the football match?'
-------------------------------------------------------------------

QueryResponse(matches=[ScoredVector(id='news_149', score=0.401341438, values=[], metadata={'text': 'Marine Expedition Finds New Species (AP) AP - Norwegian scientists who explored the deep waters of the Atlantic Ocean said Thursday their findings  #151; including what appear to be new species of fish and squid  #151; could be used to protect marine ecosystems worldwide.'}), ScoredVector(id='news_554', score=0.400781631, values=[], metadata={'text': "Science and President Bush Collide in Election Year With more than 4,000 scientists, including 48 Nobel Prize winners, having signed a statement opposing the Bush administration's use of scientific advice, this election year is seeing a new development in the uneasy relationship between science and politics."}), ScoredVector(id='news_1036', score=0.371358424, values=[], metadata={'text': "Md. Field Dig May Reach Back 16,000 Years (AP) AP - Robert D. Wall is too careful a scientist to say he's on the verge of a sensational discovery."}), Sco